<a href="https://colab.research.google.com/github/rakshachahar/flyrank-ml-internship/blob/main/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakshachahar/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb pandas

import duckdb
import pandas as pd
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except:
    HF_TOKEN = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

print("Connected Successfully")

Connected Successfully


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## Ranked Actions and Reason Codes

This action playbook prioritizes website pages using observable search-performance signals. Pages showing declining impressions, lower click-through rates, or weaker search positions are ranked for review before stable pages.

### Reason Codes

- **REFRESH_CONTENT** – The page shows declining visibility and may benefit from content updates.
- **IMPROVE_CTR** – The page receives impressions but relatively few clicks, suggesting improvements to titles or meta descriptions.
- **MONITOR** – The page currently performs consistently and should continue to be monitored rather than immediately changed.

In [ ]:
sample = con.sql(f"""
SELECT
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_clicks,
gsc_avg_position
FROM {fact_daily}
LIMIT 5000
""").df()

sample["ctr"] = sample["gsc_clicks"] / sample["gsc_impressions"].replace(0,1)

sample["Reason_Code"] = "MONITOR"

sample.loc[
    sample["gsc_impressions"] <
    sample["gsc_impressions"].median(),
    "Reason_Code"
] = "REFRESH_CONTENT"

sample.loc[
    sample["ctr"] <
    sample["ctr"].median(),
    "Reason_Code"
] = "IMPROVE_CTR"

sample = sample.sort_values(
    by="gsc_impressions",
    ascending=True
)

sample.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,Reason_Code
870,client_ff644d8251367cbb,content_3af1e8c7cda1297d,1,0,45.0,0.0,REFRESH_CONTENT
4214,client_73cda7b4e4f265ea,content_91cda0df215264ac,1,0,83.0,0.0,REFRESH_CONTENT
4215,client_73cda7b4e4f265ea,content_8f54581aaa2b6c4b,1,0,10.0,0.0,REFRESH_CONTENT
4219,client_73cda7b4e4f265ea,content_3df82be3e9deebaa,1,0,7.0,0.0,REFRESH_CONTENT
4224,client_73cda7b4e4f265ea,content_d9dcc57b99d6760c,1,0,7.0,0.0,REFRESH_CONTENT
830,client_ff644d8251367cbb,content_4aa1e125107f863c,1,0,9.0,0.0,REFRESH_CONTENT
831,client_ff644d8251367cbb,content_1b5d6f74f16c6e77,1,0,79.0,0.0,REFRESH_CONTENT
833,client_ff644d8251367cbb,content_5478b169de827906,1,0,64.0,0.0,REFRESH_CONTENT
842,client_ff644d8251367cbb,content_424bb4266b442cf7,1,0,84.0,0.0,REFRESH_CONTENT
847,client_ff644d8251367cbb,content_d95ec8461e5cc134,1,0,40.0,0.0,REFRESH_CONTENT


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended Use and Limits

This playbook is intended to support content teams when deciding which pages should be reviewed first. It provides decision-support based on historical search-performance signals and should not replace expert judgment. The recommendations are directional and should always be verified by a human before any content changes are made.

In [ ]:
print("Intended Users:")
print("- SEO Teams")
print("- Content Editors")
print("- Website Managers")

print("\nPrimary Purpose:")
print("- Prioritize content refresh")
print("- Improve CTR opportunities")
print("- Monitor page performance")

Intended Users:
- SEO Teams
- Content Editors
- Website Managers

Primary Purpose:
- Prioritize content refresh
- Improve CTR opportunities
- Monitor page performance


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human Review and No-Go List

Before acting on any recommendation, a human reviewer should verify that the page content is still relevant, accurate, and aligned with current business goals. Search-performance signals should be considered alongside content quality and user intent.

### No-Go List

The following decisions should never be fully automated:

- Publishing or deleting content.
- Changing factual or regulated information.
- Rewriting content without human review.
- Making business-critical decisions based only on model scores.

In [ ]:
review_checks = [
    "Verify content relevance",
    "Check content accuracy",
    "Review search intent",
    "Confirm business priority",
    "Approve before publishing"
]

print("Human Review Checklist")

for item in review_checks:
    print("-", item)

print("\nNo-Go Automation:")
print("- Automatic publishing")
print("- Automatic deletion")
print("- Automatic content rewriting")

Human Review Checklist
- Verify content relevance
- Check content accuracy
- Review search intent
- Confirm business priority
- Approve before publishing

No-Go Automation:
- Automatic publishing
- Automatic deletion
- Automatic content rewriting


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring and Retrain Triggers

The recommendations should be reviewed regularly as search-performance patterns change over time. Model retraining may be appropriate when search trends shift, feature distributions change noticeably, or model performance declines on newly collected data. Regular monitoring helps ensure that recommendations remain useful for decision-support.

In [ ]:
triggers = [
    "Large drop in model accuracy",
    "Major changes in search trends",
    "Feature distribution changes",
    "Regular quarterly model review"
]

print("Monitoring / Retraining Triggers\n")

for trigger in triggers:
    print("-", trigger)

Monitoring / Retraining Triggers

- Large drop in model accuracy
- Major changes in search trends
- Feature distribution changes
- Regular quarterly model review


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Exports for the Paper

The ranked action queue generated in this notebook is exported to the `work/outputs/` directory. These exported results can be reused in the research paper to support the recommendations section while keeping the workflow reproducible.

In [ ]:
import os

os.makedirs("work/outputs", exist_ok=True)

output = sample[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "Reason_Code"
    ]
]

output.to_csv(
    "work/outputs/action_playbook.csv",
    index=False
)

print("Export completed successfully.")
print("Saved to: work/outputs/action_playbook.csv")

output.head(10)

Export completed successfully.
Saved to: work/outputs/action_playbook.csv


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,Reason_Code
870,client_ff644d8251367cbb,content_3af1e8c7cda1297d,1,0,45.0,REFRESH_CONTENT
4214,client_73cda7b4e4f265ea,content_91cda0df215264ac,1,0,83.0,REFRESH_CONTENT
4215,client_73cda7b4e4f265ea,content_8f54581aaa2b6c4b,1,0,10.0,REFRESH_CONTENT
4219,client_73cda7b4e4f265ea,content_3df82be3e9deebaa,1,0,7.0,REFRESH_CONTENT
4224,client_73cda7b4e4f265ea,content_d9dcc57b99d6760c,1,0,7.0,REFRESH_CONTENT
830,client_ff644d8251367cbb,content_4aa1e125107f863c,1,0,9.0,REFRESH_CONTENT
831,client_ff644d8251367cbb,content_1b5d6f74f16c6e77,1,0,79.0,REFRESH_CONTENT
833,client_ff644d8251367cbb,content_5478b169de827906,1,0,64.0,REFRESH_CONTENT
842,client_ff644d8251367cbb,content_424bb4266b442cf7,1,0,84.0,REFRESH_CONTENT
847,client_ff644d8251367cbb,content_d95ec8461e5cc134,1,0,40.0,REFRESH_CONTENT


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.